In [4]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error

# =========================
# 1. LOAD DATA
# =========================

columns = ['user_id', 'item_id', 'rating', 'timestamp']

# Load train và test
train_data = pd.read_csv('../ml-100k/ua.base', sep='\t', names=columns)
test_data  = pd.read_csv('../ml-100k/ua.test', sep='\t', names=columns)

# Load thông tin phim
movies = pd.read_csv(
    "../ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    names=["movie_id", "movie_name"] + list(range(22))
)[["movie_id", "movie_name"]]

# Map id -> tên phim
movie_dict = dict(zip(movies.movie_id, movies.movie_name))


# =========================
# 2. TẠO USER-ITEM MATRIX
# =========================

# Matrix train: user x item
train_matrix = train_data.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

# Matrix test
test_matrix = test_data.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

# Đảm bảo test có cùng số cột với train
test_matrix = test_matrix.reindex(columns=train_matrix.columns, fill_value=0)

# Convert sang numpy
train_arr = train_matrix.values
test_arr  = test_matrix.values


# =========================
# 3. ÁP DỤNG SVD
# =========================

def apply_svd(matrix, k):
    """
    matrix: user-item matrix
    k: số latent factors
    """
    # Phân rã SVD: A = U * Sigma * V^T
    U, sigma, Vt = svds(matrix, k=k)

    # Convert sigma thành ma trận đường chéo
    sigma = np.diag(sigma)

    return U, sigma, Vt


def predict_ratings(U, sigma, Vt):
    """
    Tái tạo lại ma trận rating dự đoán
    """
    return np.dot(np.dot(U, sigma), Vt)


# =========================
# 4. TRAIN MODEL
# =========================

k = 15  # số latent features

U, sigma, Vt = apply_svd(train_arr, k)

pred_matrix = predict_ratings(U, sigma, Vt)

# Convert về DataFrame cho dễ thao tác
df_predictions = pd.DataFrame(pred_matrix,
                             index=train_matrix.index,
                             columns=train_matrix.columns)


# =========================
# 5. RECOMMEND CHO 1 USER
# =========================

def recommend_for_user(user_id, train_matrix, pred_matrix, top_k=10):
    """
    Gợi ý phim cho user
    """
    user_idx = user_id - 1  # vì index bắt đầu từ 0

    # Rating dự đoán
    predicted_ratings = pred_matrix[user_idx]

    # Những item đã rating
    rated_items = train_matrix.iloc[user_idx] > 0

    # Lấy item chưa rating
    unrated_items = np.where(rated_items == 0)[0]

    # Lấy predicted rating của item chưa rating
    scores = predicted_ratings[unrated_items]

    # Sắp xếp giảm dần
    top_items = unrated_items[np.argsort(scores)[::-1][:top_k]]

    return top_items


# Test recommend
top_items = recommend_for_user(1, train_matrix, pred_matrix)

print("Top recommendation for user 1:")
for item in top_items:
    print(movie_dict.get(item + 1, f"Movie {item+1}"))


# =========================
# 6. EVALUATION
# =========================

# Flatten matrix
y_true = test_arr.flatten()
y_pred = pred_matrix.flatten()

# Chỉ lấy phần có rating thật
mask = y_true != 0
y_true = y_true[mask]
y_pred = y_pred[mask]

# RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAE
mae = np.mean(np.abs(y_true - y_pred))


# =========================
# 7. PRECISION & RECALL @K
# =========================

def precision_recall_at_k(test_matrix, pred_matrix, k=10):
    num_users = test_matrix.shape[0]

    precision = 0
    recall = 0

    for u in range(num_users):
        actual = test_matrix.iloc[u].values
        predicted = pred_matrix[u]

        # Top K gợi ý
        top_k_items = np.argsort(predicted)[::-1][:k]

        # Item relevant (có rating trong test)
        relevant = actual[top_k_items] > 0

        precision += np.sum(relevant) / k

        # Tổng item thực sự relevant
        total_relevant = np.sum(actual > 0)

        if total_relevant > 0:
            recall += np.sum(relevant) / total_relevant

    return precision / num_users, recall / num_users


precision, recall = precision_recall_at_k(test_matrix, pred_matrix, k=10)

# F1-score
f1 = 2 * (precision * recall) / (precision + recall)


# =========================
# 8. HIT RATE
# =========================

def hit_rate_at_k(test_matrix, pred_matrix, k=10):
    num_users = test_matrix.shape[0]
    hits = 0

    for u in range(num_users):
        actual = test_matrix.iloc[u].values
        predicted = pred_matrix[u]

        top_k_items = np.argsort(predicted)[::-1][:k]

        if np.sum(actual[top_k_items] > 0) > 0:
            hits += 1

    return hits / num_users


hit_rate = hit_rate_at_k(test_matrix, pred_matrix)


# =========================
# 9. PRINT RESULT
# =========================

print("RMSE:", rmse)
print("MAE:", mae)
print("Precision@10:", precision)
print("Recall@10:", recall)
print("F1:", f1)
print("Hit Rate:", hit_rate)

Top recommendation for user 1:
Trainspotting (1996)
Close Shave, A (1995)
Schindler's List (1993)
Heathers (1989)
Groundhog Day (1993)
Batman (1989)
Stand by Me (1986)
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
E.T. the Extra-Terrestrial (1982)
One Flew Over the Cuckoo's Nest (1975)
RMSE: 2.826178335729891
MAE: 2.549554262400401
Precision@10: 0.1010604453870626
Recall@10: 0.1010604453870626
F1: 0.1010604453870626
Hit Rate: 0.559915164369035
